# Approximating Continuous Shocks

Lifecycle models need stochastic state variables — income, health, productivity —
that follow continuous distributions. Because the DP solution operates on finite
grids, these distributions must be approximated by finite Markov chains. This page
derives the formulas implemented in `lcm.shocks` and links each to its source code.

Notation follows [Sargent & Stachurski, *Dynamic Programming*,
Section 3.1.3](https://dp.quantecon.org/).

In [ ]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from jax.scipy.stats.norm import cdf as Phi  # noqa: N812
from plotly.subplots import make_subplots

from lcm.shocks._base import _gauss_hermite_normal

blue, orange, green = "#4C78A8", "#F58518", "#54A24B"

## IID Shocks

IID shocks are draws from a fixed distribution each period — no persistence.
pylcm provides `Uniform`, `Normal`, `LogNormal`, and `NormalMixture` in
`lcm.shocks.iid`. Each must specify grid points and transition probabilities
(weights). Because the draws are independent, every row of the transition matrix
$P$ is identical.

### Uniform

The simplest case: $n$ equally spaced points on $[a, b]$ with equal weights
$w_i = 1/n$.

Reference: `iid.Uniform.compute_gridpoints` (line 45 of `iid.py`) and
`compute_transition_probs` (lines 48–53).

### Normal (Gauss-Hermite Quadrature)

For $\varepsilon \sim N(\mu, \sigma^2)$, Gauss-Hermite quadrature approximates
integrals against the Gaussian weight function:

$$\int_{-\infty}^{\infty} f(x)\, e^{-x^2}\, dx
\;\approx\; \sum_{i=1}^n w_i\, f(z_i)$$

where $\{z_i\}$ and $\{w_i\}$ are the physicist's Hermite nodes and weights.

To integrate against $N(\mu, \sigma^2)$ instead, apply the affine transformation
$x_i = \mu + \sqrt{2}\,\sigma\,z_i$ and rescale the weights
$\tilde{w}_i = w_i / \sqrt{\pi}$. Then

$$\int f(x)\, \frac{1}{\sigma\sqrt{2\pi}}
e^{-(x-\mu)^2/(2\sigma^2)}\, dx
\;\approx\; \sum_{i=1}^n \tilde{w}_i\, f(x_i).$$

Reference: `_base._gauss_hermite_normal` (lines 16–29 of `_base.py`).

#### CDF-based binning (alternative)

When `gauss_hermite=False`, `Normal` uses $n$ equally spaced points spanning
$\mu \pm m\sigma$ and assigns weights by CDF binning. Define midpoints
$m_j = (x_j + x_{j+1})/2$ between consecutive grid points. Then

$$w_j = \begin{cases}
\Phi\!\left(\dfrac{m_1 - \mu}{\sigma}\right) & j = 1 \\[6pt]
\Phi\!\left(\dfrac{m_j - \mu}{\sigma}\right)
- \Phi\!\left(\dfrac{m_{j-1} - \mu}{\sigma}\right) & 1 < j < n \\[6pt]
1 - \Phi\!\left(\dfrac{m_{n-1} - \mu}{\sigma}\right) & j = n
\end{cases}$$

The first and last bins absorb the tails.

Reference: `iid.Normal.compute_transition_probs` (lines 110–125 of `iid.py`).

In [ ]:
# --- Gauss-Hermite vs equally spaced + CDF binning for N(0, 1), n = 7 ---

n = 7
mu, sigma = 0.0, 1.0

# Gauss-Hermite nodes and weights
gh_nodes, gh_weights = _gauss_hermite_normal(n, mu, sigma)

# Equally spaced nodes with CDF-based weights (n_std = 3)
n_std = 3.0
eq_nodes = jnp.linspace(mu - n_std * sigma, mu + n_std * sigma, n)
step = eq_nodes[1] - eq_nodes[0]
half_step = 0.5 * step

eq_weights = jnp.zeros(n)
eq_weights = eq_weights.at[1:].set(jnp.diff(Phi((eq_nodes + half_step - mu) / sigma)))
eq_weights = eq_weights.at[0].set(Phi((eq_nodes[0] + half_step - mu) / sigma))
eq_weights = eq_weights.at[-1].set(1 - Phi((eq_nodes[-1] - half_step - mu) / sigma))

# Side-by-side bar charts
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Gauss-Hermite", "Equally spaced + CDF binning"),
)
fig.add_trace(
    go.Bar(
        x=np.asarray(gh_nodes),
        y=np.asarray(gh_weights),
        marker_color=blue,
        width=0.15,
        name="GH weights",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=np.asarray(eq_nodes),
        y=np.asarray(eq_weights),
        marker_color=orange,
        width=0.15,
        name="CDF weights",
    ),
    row=1,
    col=2,
)
fig.update_xaxes(title_text="Node", row=1, col=1)
fig.update_xaxes(title_text="Node", row=1, col=2)
fig.update_yaxes(title_text="Weight", row=1, col=1)
fig.update_layout(height=400, width=850, showlegend=False)
fig.show()

### LogNormal

If $X \sim N(\mu, \sigma^2)$, then $Y = e^X$ is log-normal. Grid points are
$y_i = e^{x_i}$ where $\{x_i\}$ are the normal nodes (GH or equally spaced);
weights are unchanged.

Reference: `iid.LogNormal.compute_gridpoints` (lines 163–169 of `iid.py`).

In [ ]:
# --- LogNormal: GH nodes for N(0, 0.5^2) mapped to lognormal ---

mu_ln, sigma_ln = 0.0, 0.5
normal_nodes, normal_weights = _gauss_hermite_normal(n, mu_ln, sigma_ln)
lognormal_nodes = jnp.exp(normal_nodes)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=np.asarray(normal_nodes),
        y=np.asarray(normal_weights),
        mode="markers+lines",
        marker={"color": blue, "size": 8},
        line={"color": blue, "width": 1, "dash": "dot"},
        name="Normal nodes",
    )
)
fig.add_trace(
    go.Scatter(
        x=np.asarray(lognormal_nodes),
        y=np.asarray(normal_weights),
        mode="markers+lines",
        marker={"color": orange, "size": 8},
        line={"color": orange, "width": 1, "dash": "dot"},
        name="LogNormal nodes (exp transform)",
    )
)
fig.update_layout(
    title="Normal vs LogNormal grid points (GH quadrature)",
    xaxis_title="Node value",
    yaxis_title="Weight",
    height=400,
    width=700,
)
fig.show()

### NormalMixture

When the shock follows a two-component normal mixture
$\varepsilon \sim p_1 \, N(\mu_1, \sigma_1^2) + (1-p_1) \, N(\mu_2, \sigma_2^2)$,
`NormalMixture` uses equally spaced points spanning the mixture mean $\pm m$ mixture
standard deviations and assigns weights via CDF binning with the mixture CDF in
place of $\Phi$ — the same approach described in the
[Non-Normal Innovations](#non-normal-innovations-fgp-section-43) section below, but
without persistence.

Reference: `iid.NormalMixture` in `iid.py`.

## AR(1) Processes

The canonical process (Sargent & Stachurski notation):

$$X_{t+1} = \rho\, X_t + b + \nu\,\varepsilon_{t+1}, \quad
|\rho| < 1, \quad
\varepsilon_t \stackrel{\text{IID}}{\sim} N(0, 1)$$

Stationary distribution: $\psi^* = N(\mu_x,\, \sigma_x^2)$ with
$\mu_x = b/(1-\rho)$ and $\sigma_x^2 = \nu^2/(1-\rho^2)$.

The goal: construct a finite state space $X = \{x_1, \ldots, x_n\}$ and an
$n \times n$ transition matrix $P$ where
$P_{ij} = \Pr(X_{t+1} = x_j \mid X_t = x_i)$, such that the Markov chain
$(X_t)$ on $X$ has moments close to the continuous process.

| Symbol | Meaning | pylcm name |
|--------|---------|------------|
| $b$ | intercept (drift) | `mu` |
| $\nu$ | innovation std dev | `sigma` |
| $\rho$ | persistence | `rho` |
| $\mu_x = b/(1-\rho)$ | unconditional mean | — |
| $\sigma_x = \nu/\sqrt{1-\rho^2}$ | unconditional std | `std_y` in code |
| $m$ | grid half-span in SDs | `n_std` |
| $s$ | step size $(x_n - x_1)/(n-1)$ | `step` |
| $X = \{x_1,\ldots,x_n\}$ | discrete state space | grid points |
| $P$ | transition matrix | return of `compute_transition_probs` |
| $\Phi$ | standard normal CDF | `jax.scipy.stats.norm.cdf` |
| $\psi^*$ | stationary distribution | — |

In [ ]:
def ar1_moments(rho, sigma, mu):
    """Compute theoretical moments of an AR(1) process."""
    mu_x = mu / (1 - rho)
    sigma_x = sigma / jnp.sqrt(1 - rho**2)
    return mu_x, sigma_x


def conditional_density(x_prime, x_i, rho, sigma, mu):
    """PDF of X_{t+1} | X_t = x_i for a normal-innovation AR(1)."""
    cond_mean = rho * x_i + mu
    return jnp.exp(-0.5 * ((x_prime - cond_mean) / sigma) ** 2) / (
        sigma * jnp.sqrt(2 * jnp.pi)
    )

## Tauchen (1986)

Following Sargent & Stachurski, Section 3.1.3:

**1. Grid.** Choose $m$ (number of unconditional SDs) and set

$$x_n = \mu_x + m\,\sigma_x, \qquad
x_1 = \mu_x - m\,\sigma_x, \qquad
s = \frac{x_n - x_1}{n - 1}.$$

**2. Transition probabilities.** The conditional distribution of $X_{t+1}$ given
$X_t = x_i$ is $N(\rho\, x_i + b,\; \nu^2)$. Bin the probability mass at
midpoints $x_j \pm s/2$:

$$P_{ij} = \begin{cases}
\Phi\!\left(\dfrac{x_1 + s/2 - \rho\, x_i - b}{\nu}\right)
& j = 1 \\[6pt]
\Phi\!\left(\dfrac{x_j + s/2 - \rho\, x_i - b}{\nu}\right)
- \Phi\!\left(\dfrac{x_j - s/2 - \rho\, x_i - b}{\nu}\right)
& 1 < j < n \\[6pt]
1 - \Phi\!\left(\dfrac{x_n - s/2 - \rho\, x_i - b}{\nu}\right)
& j = n
\end{cases}$$

Note: the denominator is $\nu$ (innovation std), not $\sigma_x$
(unconditional std).

Reference: `ar1.Tauchen.compute_gridpoints` (lines 80–89 of `ar1.py`) and
`compute_transition_probs` (lines 91–121).

### Tauchen with Gauss-Hermite nodes

When `gauss_hermite=True`, the grid uses Gauss-Hermite quadrature nodes of
$N(\mu_x, \sigma_x^2)$ instead of equally spaced points. Transition probabilities
use CDF binning at midpoints between consecutive GH nodes (same idea, different
node placement).

Reference: `ar1.Tauchen.compute_gridpoints` (lines 82–84 of `ar1.py`) and
`compute_transition_probs` (lines 95–108).

In [ ]:
# --- Tauchen: grid and transition matrix ---

rho, sigma, mu_ar = 0.95, 0.127, 0.0
n_ar = 9
m = 3
mu_x, sigma_x = ar1_moments(rho, sigma, mu_ar)

# Grid (equally spaced, centered on unconditional mean)
x_max = m * sigma_x
x_tauchen = jnp.linspace(mu_x - x_max, mu_x + x_max, n_ar)
step_t = (2 * x_max) / (n_ar - 1)

# Transition matrix
z = x_tauchen[None, :] - rho * x_tauchen[:, None]  # z[i,j] = x[j] - rho*x[i]
upper = Phi((z + 0.5 * step_t - mu_ar) / sigma)
lower = Phi((z - 0.5 * step_t - mu_ar) / sigma)
P_tauchen = upper - lower
P_tauchen = P_tauchen.at[:, 0].set(upper[:, 0])
P_tauchen = P_tauchen.at[:, -1].set(1 - lower[:, -1])

print("Grid:", np.round(np.asarray(x_tauchen), 3))
print(f"Step size: {float(step_t):.4f}")
print(f"Row sums (should be 1): {np.round(np.asarray(P_tauchen.sum(axis=1)), 6)}")

# Plot one row of P overlaid with the true conditional density
row_idx = n_ar // 2  # middle state
x_dense = jnp.linspace(float(x_tauchen[0]) - 0.3, float(x_tauchen[-1]) + 0.3, 300)
pdf_vals = conditional_density(x_dense, x_tauchen[row_idx], rho, sigma, mu_ar)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=np.asarray(x_dense),
        y=np.asarray(pdf_vals),
        mode="lines",
        line={"color": "gray", "width": 1},
        name=f"True density given x = {float(x_tauchen[row_idx]):.2f}",
    )
)
fig.add_trace(
    go.Bar(
        x=np.asarray(x_tauchen),
        y=np.asarray(P_tauchen[row_idx] / step_t),
        width=float(step_t) * 0.9,
        marker_color=blue,
        opacity=0.6,
        name="Tauchen probabilities (scaled)",
    )
)
fig.update_layout(
    title=f"Tauchen: transition from x = {float(x_tauchen[row_idx]):.2f}"
    f" (row {row_idx} of P)",
    xaxis_title="x'",
    yaxis_title="Density / Scaled probability",
    height=400,
    width=700,
)
fig.show()

In [ ]:
# --- Tauchen: stationary distribution ---

# Compute stationary distribution by repeated matrix multiplication
psi = jnp.ones(n_ar) / n_ar
for _ in range(1000):
    psi = psi @ P_tauchen

# Theoretical stationary density: N(mu_x, sigma_x^2)
x_stat = jnp.linspace(float(x_tauchen[0]) - 0.2, float(x_tauchen[-1]) + 0.2, 300)
pdf_stat = jnp.exp(-0.5 * ((x_stat - mu_x) / sigma_x) ** 2) / (
    sigma_x * jnp.sqrt(2 * jnp.pi)
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=np.asarray(x_stat),
        y=np.asarray(pdf_stat),
        mode="lines",
        line={"color": "gray", "width": 1},
        name=f"N({float(mu_x):.1f}, {float(sigma_x):.3f}\u00b2)",
    )
)
fig.add_trace(
    go.Bar(
        x=np.asarray(x_tauchen),
        y=np.asarray(psi / step_t),
        width=float(step_t) * 0.9,
        marker_color=blue,
        opacity=0.6,
        name="Tauchen stationary dist.",
    )
)
fig.update_layout(
    title="Tauchen: stationary distribution vs N(\u03bc\u2093, \u03c3\u2093\u00b2)",
    xaxis_title="x",
    yaxis_title="Density",
    height=400,
    width=700,
)
fig.show()

### Limitations

Tauchen degrades for high persistence ($\rho \geq 0.97$) and few grid points,
because the equally spaced grid cannot capture the tails of the near-unit-root
process (FGP 2019, Tables 1–5). Rouwenhorst addresses this.

## Rouwenhorst (1995)

Following [Kopecky & Suen (2010)](https://doi.org/10.1016/j.red.2010.02.002):

**1. Grid.** Set

$$x_1 = \mu_x - \frac{\nu\sqrt{n-1}}{\sqrt{1-\rho^2}}, \qquad
x_n = \mu_x + \frac{\nu\sqrt{n-1}}{\sqrt{1-\rho^2}}$$

equally spaced. (Equivalently, the half-span is $\sigma_x\sqrt{n-1}$.)

**2. Transition matrix.** Define $q = (1+\rho)/2$. For $n = 2$:

$$P_2 = \begin{pmatrix} q & 1-q \\ 1-q & q \end{pmatrix}$$

For $n > 2$, the recursion (Kopecky & Suen, Eq. 3):

$$P_n = q \begin{pmatrix} P_{n-1} & \mathbf{0} \\
\mathbf{0}' & 0 \end{pmatrix}
+ (1-q) \begin{pmatrix} \mathbf{0} & P_{n-1} \\
0 & \mathbf{0}' \end{pmatrix}
+ (1-q) \begin{pmatrix} \mathbf{0}' & 0 \\
P_{n-1} & \mathbf{0} \end{pmatrix}
+ q \begin{pmatrix} 0 & \mathbf{0}' \\
\mathbf{0} & P_{n-1} \end{pmatrix}$$

with all rows except the first and last divided by 2 to normalize.

The closed-form expression (used in pylcm):

$$P_{ij} = \sum_k \binom{i}{k} \binom{n-1-i}{j-k}\,
q^{\,n-1-i-j+2k}\, (1-q)^{\,i+j-2k}$$

Reference: `ar1.Rouwenhorst.compute_gridpoints` (lines 157–161 of `ar1.py`) and
`compute_transition_probs` (lines 163–187).

### Exact Moment Matching

The key advantage (FGP Eqs. 9, 11 / Kopecky & Suen, Propositions 1–2):

- **Conditional mean**: $\sum_j P_{ij}\, x_j = \rho\, x_i + b$ (exact)
- **Conditional variance**: $\sum_j P_{ij}\,(x_j - \rho\, x_i - b)^2 = \nu^2$
  (exact)

These hold for any $n \geq 2$. Tauchen only approximates these.

In [ ]:
# --- Rouwenhorst: grid, P, and moment verification ---

from math import comb

# Grid
nu_r = jnp.sqrt((n_ar - 1) / (1 - rho**2)) * sigma
x_rouw = jnp.linspace(mu_x - nu_r, mu_x + nu_r, n_ar)
step_r = x_rouw[1] - x_rouw[0]

# Transition matrix (closed-form binomial)
q = (rho + 1) / 2
C = jnp.array([[comb(nn, kk) for kk in range(n_ar)] for nn in range(n_ar)])
i = jnp.arange(n_ar)[:, None, None]
j = jnp.arange(n_ar)[None, :, None]
k = jnp.arange(n_ar)[None, None, :]
valid = (k >= jnp.maximum(0, i + j - n_ar + 1)) & (k <= jnp.minimum(i, j))
k_s = jnp.where(valid, k, 0)
jmk = jnp.where(valid, j - k, 0)
terms = (
    C[i, k_s]
    * C[n_ar - 1 - i, jmk]
    * q ** (n_ar - 1 - i - j + 2 * k_s)
    * (1 - q) ** (i + j - 2 * k_s)
)
P_rouw = jnp.where(valid, terms, 0.0).sum(axis=-1)

print("Grid:", np.round(np.asarray(x_rouw), 3))
print(f"Row sums: {np.round(np.asarray(P_rouw.sum(axis=1)), 6)}")

# Verify exact conditional moments
cond_mean = P_rouw @ x_rouw  # E[X'|X=x_i]
true_cond_mean = rho * x_rouw + mu_ar
cond_var = P_rouw @ (x_rouw**2) - cond_mean**2

print("\nConditional mean errors (should be ~0):")
print(f"  max |error| = {float(jnp.max(jnp.abs(cond_mean - true_cond_mean))):.2e}")
print(f"Conditional variance (should be {sigma**2:.6f}):")
print(f"  {np.round(np.asarray(cond_var), 6)}")

# Plot one row of P
pdf_rouw = conditional_density(x_dense, x_rouw[n_ar // 2], rho, sigma, mu_ar)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=np.asarray(x_dense),
        y=np.asarray(pdf_rouw),
        mode="lines",
        line={"color": "gray", "width": 1},
        name=f"True density given x = {float(x_rouw[n_ar // 2]):.2f}",
    )
)
fig.add_trace(
    go.Bar(
        x=np.asarray(x_rouw),
        y=np.asarray(P_rouw[n_ar // 2] / step_r),
        width=float(step_r) * 0.9,
        marker_color=green,
        opacity=0.6,
        name="Rouwenhorst probabilities (scaled)",
    )
)
fig.update_layout(
    title=f"Rouwenhorst: transition from x = {float(x_rouw[n_ar // 2]):.2f}"
    f" (row {n_ar // 2} of P)",
    xaxis_title="x'",
    yaxis_title="Density / Scaled probability",
    height=400,
    width=700,
)
fig.show()

In [ ]:
# --- Rouwenhorst vs Tauchen: high-persistence case ---

rho_hi, n_hi = 0.98, 5
mu_x_hi, sigma_x_hi = ar1_moments(rho_hi, sigma, mu_ar)

# Tauchen
x_max_hi = m * sigma_x_hi
x_t_hi = jnp.linspace(mu_x_hi - x_max_hi, mu_x_hi + x_max_hi, n_hi)
step_t_hi = (2 * x_max_hi) / (n_hi - 1)
z_hi = x_t_hi[None, :] - rho_hi * x_t_hi[:, None]
upper_hi = Phi((z_hi + 0.5 * step_t_hi - mu_ar) / sigma)
lower_hi = Phi((z_hi - 0.5 * step_t_hi - mu_ar) / sigma)
P_t_hi = upper_hi - lower_hi
P_t_hi = P_t_hi.at[:, 0].set(upper_hi[:, 0])
P_t_hi = P_t_hi.at[:, -1].set(1 - lower_hi[:, -1])

# Rouwenhorst
nu_r_hi = jnp.sqrt((n_hi - 1) / (1 - rho_hi**2)) * sigma
x_r_hi = jnp.linspace(mu_x_hi - nu_r_hi, mu_x_hi + nu_r_hi, n_hi)
step_r_hi = x_r_hi[1] - x_r_hi[0]
q_hi = (rho_hi + 1) / 2
C_hi = jnp.array([[comb(nn, kk) for kk in range(n_hi)] for nn in range(n_hi)])
i_hi = jnp.arange(n_hi)[:, None, None]
j_hi = jnp.arange(n_hi)[None, :, None]
k_hi = jnp.arange(n_hi)[None, None, :]
valid_hi = (k_hi >= jnp.maximum(0, i_hi + j_hi - n_hi + 1)) & (
    k_hi <= jnp.minimum(i_hi, j_hi)
)
k_s_hi = jnp.where(valid_hi, k_hi, 0)
jmk_hi = jnp.where(valid_hi, j_hi - k_hi, 0)
terms_hi = (
    C_hi[i_hi, k_s_hi]
    * C_hi[n_hi - 1 - i_hi, jmk_hi]
    * q_hi ** (n_hi - 1 - i_hi - j_hi + 2 * k_s_hi)
    * (1 - q_hi) ** (i_hi + j_hi - 2 * k_s_hi)
)
P_r_hi = jnp.where(valid_hi, terms_hi, 0.0).sum(axis=-1)

# Stationary distributions
psi_t_hi = jnp.ones(n_hi) / n_hi
psi_r_hi = jnp.ones(n_hi) / n_hi
for _ in range(1000):
    psi_t_hi = psi_t_hi @ P_t_hi
    psi_r_hi = psi_r_hi @ P_r_hi

# Theoretical density
x_range = max(float(x_max_hi), float(nu_r_hi)) * 1.3
x_stat_hi = jnp.linspace(-x_range, x_range, 300)
pdf_stat_hi = jnp.exp(-0.5 * ((x_stat_hi - mu_x_hi) / sigma_x_hi) ** 2) / (
    sigma_x_hi * jnp.sqrt(2 * jnp.pi)
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Tauchen", "Rouwenhorst"),
)
for col, (x_grid, psi_grid, step_grid, color) in enumerate(
    [
        (x_t_hi, psi_t_hi, step_t_hi, blue),
        (x_r_hi, psi_r_hi, step_r_hi, green),
    ],
    start=1,
):
    fig.add_trace(
        go.Scatter(
            x=np.asarray(x_stat_hi),
            y=np.asarray(pdf_stat_hi),
            mode="lines",
            line={"color": "gray", "width": 1},
            name="N(\u03bc\u2093, \u03c3\u2093\u00b2)",
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Bar(
            x=np.asarray(x_grid),
            y=np.asarray(psi_grid / step_grid),
            width=float(step_grid) * 0.85,
            marker_color=color,
            opacity=0.6,
            name="Tauchen" if col == 1 else "Rouwenhorst",
        ),
        row=1,
        col=col,
    )
fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_xaxes(title_text="x", row=1, col=2)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_layout(
    title=f"Stationary distributions: \u03c1 = {rho_hi}, n = {n_hi}",
    height=400,
    width=900,
)
fig.show()

In [ ]:
# --- Conditional moment errors: Tauchen vs Rouwenhorst ---

# Tauchen conditional moments (original parameters)
cond_mean_t = P_tauchen @ x_tauchen
true_cond_mean_t = rho * x_tauchen + mu_ar
mean_err_t = jnp.abs(cond_mean_t - true_cond_mean_t)

# Rouwenhorst conditional moments
cond_mean_r = P_rouw @ x_rouw
true_cond_mean_r = rho * x_rouw + mu_ar
mean_err_r = jnp.abs(cond_mean_r - true_cond_mean_r)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=np.arange(n_ar) - 0.2,
        y=np.asarray(mean_err_t),
        width=0.35,
        marker_color=blue,
        name="Tauchen",
    )
)
fig.add_trace(
    go.Bar(
        x=np.arange(n_ar) + 0.2,
        y=np.asarray(mean_err_r),
        width=0.35,
        marker_color=green,
        name="Rouwenhorst",
    )
)
fig.update_layout(
    title="|E[X' | X = x\u1d62] \u2212 (\u03c1x\u1d62 + b)| by grid point",
    xaxis_title="Grid point index i",
    yaxis_title="Absolute error",
    yaxis_type="log",
    barmode="group",
    height=400,
    width=700,
)
fig.show()

## Non-Normal Innovations (FGP Section 4.3)

When $\varepsilon_t$ has a mixture-of-normals distribution:

$$\varepsilon_t \sim p_1 \, N(\mu_1, \sigma_1^2)
+ (1 - p_1) \, N(\mu_2, \sigma_2^2)$$

the only change to Tauchen's method is replacing $\Phi$ with the mixture CDF:

$$F_\varepsilon(x) = p_1 \, \Phi\!\left(\frac{x - \mu_1}{\sigma_1}\right)
+ (1 - p_1) \, \Phi\!\left(\frac{x - \mu_2}{\sigma_2}\right)$$

The grid spans $\pm m\,\sigma_x$ as before, with the unconditional variance
computed from the mixture:

$$\text{Var}(\varepsilon) = p_1(\sigma_1^2 + \mu_1^2)
+ (1 - p_1)(\sigma_2^2 + \mu_2^2)
- [p_1\,\mu_1 + (1 - p_1)\,\mu_2]^2$$

Reference: `_base._mixture_cdf` in `_base.py` and
`ar1.TauchenNormalMixture.compute_transition_probs` in `ar1.py`.
The IID variant `iid.NormalMixture` uses the same mixture CDF for
CDF-binning weights without persistence.

In [ ]:
# --- Mixture CDF vs normal CDF ---

# FGP-style parameters
p1, mu1, sigma1 = 0.9, 0.0, 0.1
mu2, sigma2 = -0.5, 0.3

x_cdf = jnp.linspace(-1.5, 1.0, 300)

# Mixture CDF
F_mix = p1 * Phi((x_cdf - mu1) / sigma1) + (1 - p1) * Phi((x_cdf - mu2) / sigma2)

# Normal CDF with same mean and variance
mean_eps = p1 * mu1 + (1 - p1) * mu2
var_eps = p1 * (sigma1**2 + mu1**2) + (1 - p1) * (sigma2**2 + mu2**2) - mean_eps**2
std_eps = jnp.sqrt(var_eps)
F_normal = Phi((x_cdf - mean_eps) / std_eps)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=np.asarray(x_cdf),
        y=np.asarray(F_normal),
        mode="lines",
        line={"color": "gray", "width": 1},
        name="Normal (same mean/var)",
    )
)
fig.add_trace(
    go.Scatter(
        x=np.asarray(x_cdf),
        y=np.asarray(F_mix),
        mode="lines",
        line={"color": orange, "width": 2},
        name="Mixture CDF",
    )
)
fig.update_layout(
    title="Mixture vs Normal CDF (heavier left tail)",
    xaxis_title="x",
    yaxis_title="F(x)",
    height=400,
    width=700,
)
fig.show()

In [ ]:
# --- Mixture transition matrix vs normal-innovation Tauchen ---

rho_mix, mu_mix, n_std_mix = 0.95, 0.0, 3
n_mix = 9

# Unconditional moments for mixture case
sigma_eps_sq = p1 * (sigma1**2 + mu1**2) + (1 - p1) * (sigma2**2 + mu2**2) - mean_eps**2
std_y_mix = jnp.sqrt(sigma_eps_sq / (1 - rho_mix**2))
long_run_mean_mix = (mu_mix + mean_eps) / (1 - rho_mix)
x_max_mix = n_std_mix * std_y_mix
x_mix = jnp.linspace(
    long_run_mean_mix - x_max_mix, long_run_mean_mix + x_max_mix, n_mix
)
step_mix = (2 * x_max_mix) / (n_mix - 1)
half_step_mix = 0.5 * step_mix

# Mixture transition matrix
z_mix = x_mix[None, :] - mu_mix - rho_mix * x_mix[:, None]
upper_mix = p1 * Phi((z_mix + half_step_mix - mu1) / sigma1) + (1 - p1) * Phi(
    (z_mix + half_step_mix - mu2) / sigma2
)
lower_mix = p1 * Phi((z_mix - half_step_mix - mu1) / sigma1) + (1 - p1) * Phi(
    (z_mix - half_step_mix - mu2) / sigma2
)
P_mix = upper_mix - lower_mix
P_mix = P_mix.at[:, 0].set(upper_mix[:, 0])
P_mix = P_mix.at[:, -1].set(1 - lower_mix[:, -1])

# Normal-innovation Tauchen (same unconditional std for comparison)
sigma_normal_equiv = jnp.sqrt(sigma_eps_sq)
mu_x_norm, sigma_x_norm = ar1_moments(rho_mix, sigma_normal_equiv, mu_mix)
x_max_norm = n_std_mix * sigma_x_norm
x_norm = jnp.linspace(mu_x_norm - x_max_norm, mu_x_norm + x_max_norm, n_mix)
step_norm = (2 * x_max_norm) / (n_mix - 1)
z_norm = x_norm[None, :] - rho_mix * x_norm[:, None]
upper_norm = Phi((z_norm + 0.5 * step_norm - mu_mix) / sigma_normal_equiv)
lower_norm = Phi((z_norm - 0.5 * step_norm - mu_mix) / sigma_normal_equiv)
P_norm = upper_norm - lower_norm
P_norm = P_norm.at[:, 0].set(upper_norm[:, 0])
P_norm = P_norm.at[:, -1].set(1 - lower_norm[:, -1])

# Compare one row
row_idx_mix = n_mix // 2
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=np.arange(n_mix) - 0.2,
        y=np.asarray(P_norm[row_idx_mix]),
        width=0.35,
        marker_color=blue,
        name="Normal innovations",
    )
)
fig.add_trace(
    go.Bar(
        x=np.arange(n_mix) + 0.2,
        y=np.asarray(P_mix[row_idx_mix]),
        width=0.35,
        marker_color=orange,
        name="Mixture innovations",
    )
)
fig.update_layout(
    title=f"Transition probabilities from middle state (row {row_idx_mix})",
    xaxis_title="Target state index j",
    yaxis_title="P[i, j]",
    barmode="group",
    height=400,
    width=700,
)
fig.show()

## Summary

The Markov chain approximations are fully specified by grid points and a transition
matrix $P$. pylcm computes these via `compute_gridpoints` and
`compute_transition_probs` on each `_ShockGrid` subclass.

**References**

- Fella, G., Gallipoli, G. & Pan, J. (2019). Markov-chain approximations for
  life-cycle models. *Review of Economic Dynamics*, 34, 183–201.
  [doi:10.1016/j.red.2019.01.001](https://doi.org/10.1016/j.red.2019.01.001)
- Kopecky, K. A. & Suen, R. M. H. (2010). Finite state Markov-chain approximations
  to highly persistent processes. *Review of Economic Dynamics*, 13(3), 701–714.
  [doi:10.1016/j.red.2010.02.002](https://doi.org/10.1016/j.red.2010.02.002)
- Rouwenhorst, K. G. (1995). Asset pricing implications of equilibrium business
  cycle models. In *Frontiers of Business Cycle Research* (pp. 294–330). Princeton
  University Press.
- Sargent, T. J. & Stachurski, J. *Dynamic Programming*. QuantEcon.
  [dp.quantecon.org](https://dp.quantecon.org/)
- Tauchen, G. (1986). Finite state Markov-chain approximations to univariate and
  vector autoregressions. *Economics Letters*, 20(2), 177–181.